In [1]:
!pip install z3-solver

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.8/38.8 MB 14.9 MB/s  0:00:02 eta 0:00:01


In [7]:
from z3 import (
    Bool, BoolRef, ExprRef, Implies, Not, And, Or, Solver,
    is_not, is_and, is_or, is_implies, is_eq, sat
)
from typing import Dict, Tuple, List
import itertools

# PropositionalSymbol: a bool variable. e.g Bool("Rain")
PropositionalSymbol = BoolRef

# Formula: Any boolean Z3 expression constructed with connectives
Formula = ExprRef

# Model: An assignment mapping each symbol to True or False
Model = Dict[PropositionalSymbol, bool]

In [6]:
# The Interpretation Function I(f, w)

# Given a formula and a model, recursively computes whether model satisfies formula.
def I(f: Formula, w: Model) -> bool:
    # If f is a PropositionalSymbol (bool variable), return its value in w
    if f.num_args() == 0:
        result = w[f]
    # If f is Not(PropositionalSymbol):
    elif f.num_args() == 1:
        if is_not(f):
            result = not I(f.arg(0), w)
        else:
            raise ValueError(f"Unsupported formula: {f}")
    elif f.num_args() == 2:
        a, b = I(f.arg(0), w), I(f.arg(1), w)
        if is_and(f):
            result = a and b
        elif is_or(f):
            result = a or b
        elif is_eq(f):
            result = (a == b)
        elif is_implies(f):
            result = (not a) or b
        else:
            raise ValueError(f"Unsupported formula: {f}")
    else:
        raise ValueError(f"Unsupported formula: {f}, num args: {f.num_args()}")
    return result

# Test Interpretation Function
Rain = Bool("Rain")
Wet = Bool("Wet")
f_test = Implies(Rain, Wet)
w_test1 = {Rain: True, Wet: False}
w_test2 = {Rain: True, Wet: True}
print("Test 1 (Rain=True, Wet=False):", I(f_test, w_test1)) # Expected: False
print("Test 2 (Rain=True, Wet=True) :", I(f_test, w_test2)) # Expected: True

A = Bool("A")
B = Bool("B")
C = Bool("C")
f_test1 = And(Not(A), B) == C  # (¬A ∧ B) ↔ C
w_test3 = {A: True, B: True, C: False}
print("Test 3 (A=True, B=True, C: False) :", I(f_test1, w_test3)) # Expected: True

Test 1 (Rain=True, Wet=False): False
Test 2 (Rain=True, Wet=True) : True
Test 3 (A=True, B=True, C: False) : True


In [8]:
# Model Enumeration and Knowledge Base Mechanics

def get_models(f: Formula, symbols: List[PropositionalSymbol]) -> List[Model]:
    models = []
    # Enumerate all 2^N possible truth-value combinations
    for values in itertools.product([True, False], repeat=len(symbols)):
        w = dict(zip(symbols, values))
        if I(f, w):
            models.append(w)
    return models

# Test Model Enumeration
symbols = [Rain, Wet]
f_test = Or(Rain, Wet)
models = get_models(f_test, symbols)

print(f"Total possible worlds:{2**len(symbols)}")
print(f"Number of satisfying models for (Rain | Wet):{len(models)}")


Total possible worlds:4
Number of satisfying models for (Rain | Wet):3


In [15]:
# Knowledge Base Relationships (Entailment, Contradiction, Contingency)

def to_formula(kb: list[Formula]) -> Formula:
    f = kb[0]
    for g in kb[1:]:
        f = And(f, g)
    return f

# Models of the knowledge base
def M(kb: list[Formula], symbols: list[PropositionalSymbol]) -> list[Model]:
    if not kb: # if kb is empty, return all 2^n combinations
        return [dict(zip(symbols, values)) for values in itertools.product([True, False], repeat=len(symbols))]
    return get_models(to_formula(kb), symbols)

# Entailment
def entails(kb: list[Formula], f: Formula, symbols: list[PropositionalSymbol]) -> bool:
    old_models = M(kb, symbols)
    new_models = M(kb + [f], symbols)
    return old_models == new_models

# Contradiction
def contradicts(kb: list[Formula], f: Formula, symbols: list[PropositionalSymbol]) -> bool:
    new_models = M(kb + [f], symbols)
    return new_models == []

# Contingency
def contingent(kb: list[Formula], f: Formula, symbols: list[PropositionalSymbol]) -> bool:
    old_models = M(kb, symbols)
    new_models = M(kb + [f], symbols)
    return new_models != old_models and new_models != []

# Test
kb_sample = [Rain, Implies(Rain, Wet)]
assert entails(kb_sample, Rain, symbols) == True
assert contradicts(kb_sample, Not(Wet), symbols) == True
assert contingent([Rain], Wet, symbols) == True
print("Models of empty kb:", M([], symbols))

Models of empty kb: [{Rain: True, Wet: True}, {Rain: True, Wet: False}, {Rain: False, Wet: True}, {Rain: False, Wet: False}]


In [20]:
# Ask / Tell Agent Interface

def ask(kb: list[Formula], f: Formula, symbols: list[PropositionalSymbol]) -> str:
    old_models = M(kb, symbols)
    new_models = M(kb + [f], symbols)
    if new_models == old_models: # entailment
        return "Yes"
    elif new_models == []: #contradiction
        return "No"
    else:
        return "I don't know" #contingency

def tell(kb: list[Formula], f: Formula, symbols: list[PropositionalSymbol]):
    old_models = M(kb, symbols)
    new_models = M(kb + [f], symbols)
    if new_models == old_models: # entailment
        return kb, "I already knew it."
    elif new_models == []: #contradiction
        return kb, "I don't buy it."
    else:
        return kb + [f], "I learned something new." #contingency

# Test ask and tell
kb_test = []
response = ask(kb_test, Wet, symbols)
print("1. Initial state, ask if it's wet:", response)
kb_test, response = tell(kb_test, Implies(Rain, Wet), symbols)
print("2. Tell (Rain -> Wet):", response)
kb_test, response = tell(kb_test, Rain, symbols)
print("3. Tell Rain:", response)
response = ask(kb_test, Wet, symbols)
print("4. Ask if it's wet now:", response)


1. Initial state, ask if it's wet: I don't know
2. Tell (Rain -> Wet): I learned something new.
3. Tell Rain: I learned something new.
4. Ask if it's wet now: Yes


In [24]:
# Reduction to Satisfiability using Z3 SAT Solver

def fast_ask(kb: list[Formula], f: Formula) -> str:
    # If KB U {f} = UNSAT, then contradiction
    solver1 = Solver()
    for i in kb + [f]:
        solver1.add(i)
    if solver1.check() != sat:
        return "No"

     # If KB U {Not(f)} = UNSAT, then entailment
    solver2 = Solver()
    for i in kb + [Not(f)]:
        solver2.add(i)
    result = solver2.check()
    if  result != sat:
        return "Yes"

    return "I don't know." 

# Testing fast_ask
kb_fast = [Rain, Implies(Rain, Wet)]

print("Fast Ask (Wet)    :", fast_ask(kb_fast, Wet))       # Expected: Yes
print("Fast Ask (Not(Wet)):", fast_ask(kb_fast, Not(Wet)))  # Expected: No
    

Fast Ask (Wet)    : Yes
Fast Ask (Not(Wet)): No


In [25]:
# Forward Inference Engine using Modus Ponens

# Derives all possible atomic propositions by repeatedly applying Modus Ponens.
# General modus ponens: p, p → q ⊢ q
def forward_inference(kb: list[Formula]) -> list[Formula]:
    kb_derived = set(kb)
    changed = True

    while changed:
        changed = False
        new_formulas = set()
        for f1 in kb_derived:
            for f2 in kb_derived:
                if is_implies(f2):
                    premise, conclusion = f2.arg(0), f2.arg(1)
                    if f1 == premise and conclusion not in kb_derived:
                        new_formulas.add(conclusion)
        if new_formulas:
            kb_derived.update(new_formulas)
            changed = True
    return list(kb_derived)

# Test Forward Inference
Slippery = Bool("Slippery")
kb_rules = [
    Rain,                         # Fact 1: Rain
    Implies(Rain, Wet),           # Rule 1: Rain -> Wet
    Implies(Wet, Slippery)        # Rule 2: Wet -> Slippery
]

derived_facts = forward_inference(kb_rules)
print("Initial KB Facts & Rules:", kb_rules)
print("All Derived Formulas via Forward Inference:", derived_facts)


Initial KB Facts & Rules: [Rain, Implies(Rain, Wet), Implies(Wet, Slippery)]
All Derived Formulas via Forward Inference: [Implies(Rain, Wet), Rain, Implies(Wet, Slippery), Slippery, Wet]
